In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
device

device(type='cuda')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
trainDataset = datasets.MNIST(root="", download = True, transform=transform)
trainDataLoader = DataLoader(trainDataset,shuffle=True, batch_size=64)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 477kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.40MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.78MB/s]


In [ ]:
testDataset = datasets.MNIST(root="", download = False, transform=transform)
testDataLoader = DataLoader(testDataset,shuffle=True, batch_size=64)

In [ ]:
from torch.nn.modules.activation import ReLU
from torch.nn.modules.linear import Linear
class CnnModel(nn.Module):
  def __init__(self, Output):
    super().__init__()

    self.Model = nn.Sequential(
        nn.Conv2d(1, out_channels=32, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32, out_channels=64, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Flatten(),

        nn.Linear(64*7*7, 128),
        nn.ReLU(),
        nn.Linear(128, Output),
    )

  def forward(self, x):
    return self.Model(x)

In [ ]:
cnnModel = CnnModel(len(trainDataset.classes)).to(device)

In [ ]:
criteria = nn.CrossEntropyLoss()

In [ ]:
cnnOptimizer = optim.Adam(cnnModel.parameters(), lr=0.001)

In [ ]:
epochs = 10

In [ ]:
for epoch in range(epochs):
  totalLoss = 0.0
  batchSize = 0
  for img, label in trainDataLoader:

    batchSize +=1
    img = img.to(device)
    label = label.to(device)

    op = cnnModel(img)

    loss = criteria(op, label)

    totalLoss += loss.item()

    cnnOptimizer.zero_grad()
    loss.backward()
    cnnOptimizer.step()

  print(f"Epoch: {epoch+1}/{epochs}, Loss: {(totalLoss/batchSize):.2}")


Epoch: 1/10, Loss: 0.15512672746315725
Epoch: 2/10, Loss: 0.04560990848760706
Epoch: 3/10, Loss: 0.031936140569075365
Epoch: 4/10, Loss: 0.022447594877255252
Epoch: 5/10, Loss: 0.018532790820039724
Epoch: 6/10, Loss: 0.013659844587554287
Epoch: 7/10, Loss: 0.010683396836265641
Epoch: 8/10, Loss: 0.00879147747776387
Epoch: 9/10, Loss: 0.008685556365986677
Epoch: 10/10, Loss: 0.007071168700275342


In [ ]:
cnnModel.eval()

total = 0
correct = 0
with torch.no_grad():
  for img, label in trainDataLoader:

    img = img.to(device)
    label = label.to(device)

    op = cnnModel(img)
    _,prediction = torch.max(op,1)

    correct += (prediction==label).sum().item()
    total += label.size(0)

print(f"Accuracy: {(correct/total*100):.1}%")

Accuracy: 1e+02%
